In [1]:
"""
=============================================================================
MODEL 5 — LightGBM Threat-Level Classifier
=============================================================================
Dataset   : cybint_clean_train.csv  (40 000 rows, 17 cols after cleaning)
Task      : 4-class classification → 0=LOW  1=MEDIUM  2=HIGH  3=CRITICAL
Regions   : india-pak | india-china | india-bangladesh | india-arunachal
-----------------------------------------------------------------------------
Pipeline sections
  1.  Imports & reproducibility
  2.  Load & quick audit
  3.  Feature engineering   ← heavily commented
  4.  Target-encoding (CV-safe)
  5.  Train / validation split
  6.  Baseline LightGBM (default params)
  7.  Optuna hyper-parameter search
  8.  Final model + threshold search
  9.  Evaluation (confusion matrix, per-class metrics, calibration)
  10. Feature importance (SHAP + built-in gain)
  11. Save artefacts
=============================================================================
"""

'\n=============================================================================\nMODEL 5 — LightGBM Threat-Level Classifier\n=============================================================================\nDataset   : cybint_clean_train.csv  (40 000 rows, 17 cols after cleaning)\nTask      : 4-class classification → 0=LOW  1=MEDIUM  2=HIGH  3=CRITICAL\nRegions   : india-pak | india-china | india-bangladesh | india-arunachal\n-----------------------------------------------------------------------------\nPipeline sections\n  1.  Imports & reproducibility\n  2.  Load & quick audit\n  3.  Feature engineering   ← heavily commented\n  4.  Target-encoding (CV-safe)\n  5.  Train / validation split\n  6.  Baseline LightGBM (default params)\n  7.  Optuna hyper-parameter search\n  8.  Final model + threshold search\n  9.  Evaluation (confusion matrix, per-class metrics, calibration)\n  10. Feature importance (SHAP + built-in gain)\n  11. Save artefacts\n============================================

In [2]:
# ─── 1. IMPORTS & REPRODUCIBILITY ──────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib  import Path
from datetime import datetime

# ML utilities
from sklearn.model_selection  import StratifiedKFold, train_test_split
from sklearn.preprocessing    import LabelEncoder
from sklearn.metrics          import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, accuracy_score
)
from sklearn.calibration      import calibration_curve
from sklearn.inspection       import permutation_importance

import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    import shap                          # optional but strongly recommended
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("[WARN] shap not installed — SHAP plots will be skipped.")

# Reproducibility seed used everywhere
SEED   = 42
np.random.seed(SEED)

# ─── PATHS ──────────────────────────────────────────────────────────────────
DATA_PATH   = Path("/Users/amitsingh/ML_Projects/WarfareAI/datasets/cybint_clean_train.csv")   # adjust if running from CLI
OUTPUT_DIR  = Path("model5_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Class mapping (for readable labels in plots)
LABEL_MAP = {0: "LOW", 1: "MEDIUM", 2: "HIGH", 3: "CRITICAL"}

In [3]:
# ============================================================================
# 2. LOAD & QUICK AUDIT
# ============================================================================
print("=" * 60)
print("Loading data …")
df = pd.read_csv(DATA_PATH)

print(f"  Shape        : {df.shape}")
print(f"  Null counts  : {df.isnull().sum().sum()}")
print(f"  Class counts :\n{df['threat_label_int'].value_counts().sort_index()}\n")

# The raw 10 simulator features already present in the CSV:
#   n_intrusions, n_jamming, n_malware          ← raw event counts
#   weighted_cyber_signal                        ← intrusions×2 + jamming×1.5 + malware×2.5
#   events_near_border                           ← geo-proximity count
#   n_active_scenarios                           ← concurrent scenario count
#   hour, day, month, dayofweek                  ← temporal primitives
#   hour_sin/cos, month_sin/cos                  ← cyclic encodings (already done)
#   region, region_code                          ← categorical border region

Loading data …
  Shape        : (40000, 17)
  Null counts  : 0
  Class counts :
threat_label_int
0    11068
1    12015
2    10967
3     5950
Name: count, dtype: int64



In [4]:
# ============================================================================
# 3. FEATURE ENGINEERING
# ============================================================================
print("Engineering features …")

fe = df.copy()      # working frame — never modify the original

# ── 3a. AGGREGATE EVENT COUNTS ──────────────────────────────────────────────

# Total raw cyber events in the window; acts as a load indicator.
fe["total_events"] = fe["n_intrusions"] + fe["n_jamming"] + fe["n_malware"]

# Maximum of any single event type — captures spikes that averages miss.
fe["max_event_type"] = fe[["n_intrusions", "n_jamming", "n_malware"]].max(axis=1)

# Minimum event type — low values here mean one vector is quieter (asymmetry).
fe["min_event_type"] = fe[["n_intrusions", "n_jamming", "n_malware"]].min(axis=1)

# Std of the three counts — high std = uneven attack surface.
fe["event_std"] = fe[["n_intrusions", "n_jamming", "n_malware"]].std(axis=1)

# Range = max − min: another asymmetry indicator.
fe["event_range"] = fe["max_event_type"] - fe["min_event_type"]

# ── 3b. RATIO / PROPORTION FEATURES ─────────────────────────────────────────
# Division by (total_events + ε) prevents div-by-zero on all-zero windows.
EPS = 1e-9

# Share of intrusions in the threat mix.
# High → attacker focused on direct network penetration.
fe["intrusion_ratio"] = fe["n_intrusions"] / (fe["total_events"] + EPS)

# Share of malware — highest weight in weighted_cyber_signal (×2.5).
# Elevated ratio often co-occurs with CRITICAL labels.
fe["malware_ratio"] = fe["n_malware"] / (fe["total_events"] + EPS)

# Share of jamming — signals spectrum/electronic warfare component.
fe["jamming_ratio"] = fe["n_jamming"] / (fe["total_events"] + EPS)

# Border proximity ratio: what fraction of all events happened near the LOC/LAC.
fe["border_density"] = fe["events_near_border"] / (fe["total_events"] + EPS)

# Normalised signal per event: effective "average severity per event".
# Useful when raw counts are low but weighted_cyber_signal is still high.
fe["signal_per_event"] = fe["weighted_cyber_signal"] / (fe["total_events"] + EPS)

# ── 3c. INTERACTION FEATURES ─────────────────────────────────────────────────

# Active scenarios amplify any existing cyber signal —
# product captures the joint escalation effect.
fe["scenario_signal_interaction"] = fe["n_active_scenarios"] * fe["weighted_cyber_signal"]

# Border events × signal: a strong signal *near the border* is more dangerous
# than the same signal in the interior.
fe["border_signal_interaction"] = fe["events_near_border"] * fe["weighted_cyber_signal"]

# Intrusion × malware: both vectors active simultaneously indicates a
# multi-stage campaign (recon → payload).
fe["intrusion_malware_product"] = fe["n_intrusions"] * fe["n_malware"]

# Malware × jamming: electronic disruption combined with malware deployment
# is a hallmark of hybrid-warfare scenarios.
fe["malware_jamming_product"] = fe["n_malware"] * fe["n_jamming"]

# Border proximity × active scenarios: coordinated field activity
# coinciding with open scenarios.
fe["border_scenario_interaction"] = fe["events_near_border"] * fe["n_active_scenarios"]

# Scenarios × total events: overall load when multiple scenarios are active.
fe["scenario_total_interaction"] = fe["n_active_scenarios"] * fe["total_events"]

# ── 3d. DOMINANCE / IMBALANCE FLAGS ─────────────────────────────────────────

# How dominant is malware relative to the other two vectors?
# > 1 means malware count exceeds the sum of intrusions + jamming.
fe["malware_dominance"] = fe["n_malware"] / (fe["n_intrusions"] + fe["n_jamming"] + EPS)

# Similar for intrusions.
fe["intrusion_dominance"] = fe["n_intrusions"] / (fe["n_malware"] + fe["n_jamming"] + EPS)

# ── 3e. THRESHOLD-BASED BINARY FLAGS ─────────────────────────────────────────

# Any single high-severity indicator tripped?
# (Thresholds set at ≥ 75th-percentile of training distribution)
fe["flag_high_intrusion"] = (fe["n_intrusions"] >= 6).astype(int)   # p75 = 6
fe["flag_high_malware"]   = (fe["n_malware"]    >= 4).astype(int)   # p75 = 4
fe["flag_high_jamming"]   = (fe["n_jamming"]    >= 4).astype(int)   # p75 = 4

# Composite: at least two high-severity flags active together.
fe["multi_vector_flag"] = (
    fe["flag_high_intrusion"] + fe["flag_high_malware"] + fe["flag_high_jamming"] >= 2
).astype(int)

# Scenario is active AND border events are elevated — physical + cyber co-activation.
fe["active_border_flag"] = (
    (fe["n_active_scenarios"] >= 1) & (fe["events_near_border"] >= 5)
).astype(int)

# ── 3f. TEMPORAL FEATURES ────────────────────────────────────────────────────
# hour_sin / hour_cos / month_sin / month_cos already exist in the CSV.
# We derive higher-level temporal context on top.

# Quarter of the year (1–4): captures seasonal threat patterns
# (e.g., higher activity near India–China in post-monsoon months).
fe["quarter"] = ((fe["month"] - 1) // 3 + 1).astype(int)

# Cyclic quarter encoding (avoids ordinal assumption quarter 4 → quarter 1 wrap).
fe["quarter_sin"] = np.sin(2 * np.pi * fe["quarter"] / 4)
fe["quarter_cos"] = np.cos(2 * np.pi * fe["quarter"] / 4)

# Is it a night-time window? (23:00–05:00 local)
# Infiltration and cyber ops are often launched at night.
fe["is_night"] = ((fe["hour"] >= 23) | (fe["hour"] <= 5)).astype(int)

# Peak operational hours (08:00–18:00): high operator tempo.
fe["is_peak_hour"] = ((fe["hour"] >= 8) & (fe["hour"] <= 18)).astype(int)

# Weekend flag (0=Mon … 6=Sun; weekend = Sat/Sun).
fe["is_weekend"] = (fe["dayofweek"] >= 5).astype(int)

# Start/end of week (Mon=0, Fri=4): ops may ramp up / wind down.
fe["is_monday"]  = (fe["dayofweek"] == 0).astype(int)
fe["is_friday"]  = (fe["dayofweek"] == 4).astype(int)

# Mid-month flag: payrolls, procurement cycles, admin windows cluster here.
fe["is_mid_month"] = ((fe["day"] >= 12) & (fe["day"] <= 20)).astype(int)

# Temporal × signal interaction: night + high signal = stealthy escalation.
fe["night_signal"] = fe["is_night"] * fe["weighted_cyber_signal"]

# ── 3g. LOG-TRANSFORM FEATURES ──────────────────────────────────────────────
# LightGBM is tree-based so log transforms are not strictly necessary,
# but they improve split quality on right-skewed counts.

fe["log_weighted_signal"]   = np.log1p(fe["weighted_cyber_signal"])
fe["log_events_near_border"] = np.log1p(fe["events_near_border"])
fe["log_total_events"]      = np.log1p(fe["total_events"])

# ── 3h. REGION-LEVEL AGGREGATED STATS (training-set safe) ───────────────────
# Per-region mean / std of the signal — captures baseline threat level
# for each border corridor.  These are computed on the FULL training set
# here; during CV they must be recomputed on the fold-train split to
# prevent leakage (see section 4 for the CV-safe target-encoding).

region_signal_stats = (
    fe.groupby("region")["weighted_cyber_signal"]
    .agg(region_mean_signal="mean", region_std_signal="std")
    .reset_index()
)
fe = fe.merge(region_signal_stats, on="region", how="left")

# How much does this window deviate from its region's baseline?
fe["signal_vs_region_mean"] = (
    fe["weighted_cyber_signal"] - fe["region_mean_signal"]
) / (fe["region_std_signal"] + EPS)

# Per-region mean border events.
region_border_stats = (
    fe.groupby("region")["events_near_border"]
    .agg(region_mean_border="mean")
    .reset_index()
)
fe = fe.merge(region_border_stats, on="region", how="left")
fe["border_vs_region_mean"] = fe["events_near_border"] - fe["region_mean_border"]

print(f"  Features after engineering : {fe.shape[1]}")

Engineering features …
  Features after engineering : 58


In [5]:
# ============================================================================
# 4. TARGET ENCODING — CV-SAFE (Leave-One-Out style inside folds)
# ============================================================================
# Standard target encoding leaks the label into the feature.
# We use 5-fold OOF encoding so training rows never see their own target.

print("Computing OOF target encoding …")

TARGET          = "threat_label_int"
SKF             = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# We'll add one column per class: P(class=k | region).
for cls in range(4):
    col_name = f"region_te_cls{cls}"
    fe[col_name] = np.nan
    binary_target = (fe[TARGET] == cls).astype(float)

    for fold_i, (tr_idx, val_idx) in enumerate(SKF.split(fe, fe[TARGET])):
        # Compute encoding from training fold rows only
        enc = (
            fe.iloc[tr_idx]
            .groupby("region")[binary_target.name if hasattr(binary_target, "name") else TARGET]
            .apply(lambda x: (fe.iloc[x.index][TARGET] == cls).mean())   # region → P(cls)
        )
        # Compute correctly using a helper frame
        tmp = fe.iloc[tr_idx][["region"]].copy()
        tmp["y"] = binary_target.iloc[tr_idx].values
        enc_map = tmp.groupby("region")["y"].mean()

        # Fill in the validation fold
        fe.loc[fe.index[val_idx], col_name] = fe.iloc[val_idx]["region"].map(enc_map)

    # Any NaN (unseen region in fold) → global mean
    global_mean = binary_target.mean()
    fe[col_name].fillna(global_mean, inplace=True)

print("  Target encoding done.")

Computing OOF target encoding …
  Target encoding done.


In [6]:
# ============================================================================
# 5. TRAIN / VALIDATION SPLIT
# ============================================================================

# Drop columns that are either the target or would cause direct leakage.
DROP_COLS = [
    "threat_label_int",   # target
    "region",             # raw string already encoded via region_code + target enc
    # weighted_cyber_signal kept — it IS an input feature per spec,
    # and the simulator uses the same formula at inference time.
]

X = fe.drop(columns=DROP_COLS)
y = fe[TARGET]

print(f"\nFinal feature matrix : {X.shape}")
print(f"Feature list         : {X.columns.tolist()}\n")

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size    = 0.15,
    stratify     = y,
    random_state = SEED
)
print(f"Train : {X_train.shape}  |  Val : {X_val.shape}")

# LightGBM datasets (used for early-stopping during tuning)
dtrain = lgb.Dataset(X_train, label=y_train)
dval   = lgb.Dataset(X_val,   label=y_val,   reference=dtrain)


Final feature matrix : (40000, 60)
Feature list         : ['n_intrusions', 'n_jamming', 'n_malware', 'weighted_cyber_signal', 'events_near_border', 'n_active_scenarios', 'hour', 'day', 'month', 'dayofweek', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'region_code', 'total_events', 'max_event_type', 'min_event_type', 'event_std', 'event_range', 'intrusion_ratio', 'malware_ratio', 'jamming_ratio', 'border_density', 'signal_per_event', 'scenario_signal_interaction', 'border_signal_interaction', 'intrusion_malware_product', 'malware_jamming_product', 'border_scenario_interaction', 'scenario_total_interaction', 'malware_dominance', 'intrusion_dominance', 'flag_high_intrusion', 'flag_high_malware', 'flag_high_jamming', 'multi_vector_flag', 'active_border_flag', 'quarter', 'quarter_sin', 'quarter_cos', 'is_night', 'is_peak_hour', 'is_weekend', 'is_monday', 'is_friday', 'is_mid_month', 'night_signal', 'log_weighted_signal', 'log_events_near_border', 'log_total_events', 'region_mean_sign

In [7]:
# ============================================================================
# 6. BASELINE LIGHTGBM (sanity check — default params)
# ============================================================================
print("\n── Baseline LightGBM ──")

BASE_PARAMS = dict(
    objective        = "multiclass",
    num_class        = 4,
    metric           = "multi_logloss",
    num_leaves       = 31,
    learning_rate    = 0.1,
    n_estimators     = 500,
    random_state     = SEED,
    verbose          = -1,
    n_jobs           = -1,
)

base_model = lgb.LGBMClassifier(**BASE_PARAMS)
base_model.fit(
    X_train, y_train,
    eval_set              = [(X_val, y_val)],
    callbacks             = [lgb.early_stopping(50, verbose=False),
                              lgb.log_evaluation(period=-1)],
)

base_preds = base_model.predict(X_val)
base_f1    = f1_score(y_val, base_preds, average="macro")
base_acc   = accuracy_score(y_val, base_preds)
print(f"  Baseline  macro-F1 : {base_f1:.4f}   accuracy : {base_acc:.4f}")



── Baseline LightGBM ──
  Baseline  macro-F1 : 0.9063   accuracy : 0.9030


In [8]:
# ============================================================================
# 7. OPTUNA HYPER-PARAMETER SEARCH
# ============================================================================
print("\n── Optuna search (100 trials) ──")

def objective(trial: optuna.Trial) -> float:
    """Optimise macro-F1 on the held-out validation split."""
    params = dict(
        objective          = "multiclass",
        num_class          = 4,
        metric             = "multi_logloss",
        verbosity          = -1,
        random_state       = SEED,
        n_jobs             = -1,

        # Tree structure
        num_leaves         = trial.suggest_int("num_leaves",  20, 512),
        max_depth          = trial.suggest_int("max_depth",    3,  15),
        min_child_samples  = trial.suggest_int("min_child_samples", 5, 100),
        min_child_weight   = trial.suggest_float("min_child_weight", 1e-4, 10.0, log=True),

        # Regularisation
        reg_alpha          = trial.suggest_float("reg_alpha",   1e-8, 10.0, log=True),
        reg_lambda         = trial.suggest_float("reg_lambda",  1e-8, 10.0, log=True),

        # Sampling (reduces variance, speeds training)
        subsample          = trial.suggest_float("subsample",         0.5, 1.0),
        subsample_freq     = trial.suggest_int("subsample_freq",       1,   5),
        colsample_bytree   = trial.suggest_float("colsample_bytree",  0.4, 1.0),

        # Learning rate + boosting
        learning_rate      = trial.suggest_float("learning_rate",  0.01, 0.3, log=True),
        n_estimators       = trial.suggest_int("n_estimators",     200, 3000),

        # Extra gradient path options
        path_smooth        = trial.suggest_float("path_smooth",    0.0, 2.0),
        extra_trees        = trial.suggest_categorical("extra_trees", [True, False]),
    )

    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set  = [(X_val, y_val)],
        callbacks = [lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(period=-1)],
    )
    preds = model.predict(X_val)
    return f1_score(y_val, preds, average="macro")


study = optuna.create_study(
    direction   = "maximize",
    sampler     = optuna.samplers.TPESampler(seed=SEED),
    pruner      = optuna.pruners.MedianPruner(n_warmup_steps=10),
)
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\n  Best trial macro-F1 : {study.best_value:.4f}")
print(f"  Best params         : {study.best_params}")




── Optuna search (100 trials) ──


Best trial: 44. Best value: 0.91077: 100%|██████████| 100/100 [10:13<00:00,  6.13s/it]


  Best trial macro-F1 : 0.9108
  Best params         : {'num_leaves': 253, 'max_depth': 4, 'min_child_samples': 95, 'min_child_weight': 2.8882138745840757, 'reg_alpha': 8.411648809420972e-07, 'reg_lambda': 0.0010055499794881346, 'subsample': 0.905154659415798, 'subsample_freq': 4, 'colsample_bytree': 0.6142644153146469, 'learning_rate': 0.07132274318632818, 'n_estimators': 2330, 'path_smooth': 0.9110982628039396, 'extra_trees': True}


In [9]:
# ============================================================================
# 8. FINAL MODEL WITH BEST PARAMS
# ============================================================================
print("\n── Training final model ──")

best_params = dict(
    objective        = "multiclass",
    num_class        = 4,
    metric           = "multi_logloss",
    verbosity        = -1,
    random_state     = SEED,
    n_jobs           = -1,
    class_weight     = "balanced",   # handles mild class imbalance (class 3 ≈ 15%)
    **study.best_params
)

final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(
    X_train, y_train,
    eval_set  = [(X_val, y_val)],
    callbacks = [lgb.early_stopping(100, verbose=True),
                 lgb.log_evaluation(period=50)],
)

# ── Predict probabilities and hard labels ───────────────────────────────────
y_prob = final_model.predict_proba(X_val)      # shape (N, 4)
y_pred = final_model.predict(X_val)

# ── Per-class optimal threshold search (maximise macro-F1) ──────────────────
# Default decision: argmax(prob).  For imbalanced classes, custom thresholds
# can push recall up for the under-represented CRITICAL class.

from itertools import product as cartesian_product

def search_thresholds(proba: np.ndarray, y_true: np.ndarray,
                      grid: np.ndarray = np.arange(0.1, 0.9, 0.05)) -> np.ndarray:
    """
    Grid-search per-class thresholds that maximise macro-F1.
    Strategy: one threshold per class; label = argmax over
    (prob - threshold) to bias toward / against specific classes.
    """
    best_f1         = 0.0
    best_thresholds = np.array([0.5] * 4)

    # Only search thresholds for the minority CRITICAL class (class 3)
    # to keep the search tractable — classes 0-2 stay at 0.5.
    for t3 in grid:
        adjusted = proba.copy()
        adjusted[:, 3] *= (0.5 / t3 + EPS)    # lower effective threshold for class 3
        preds_tmp = adjusted.argmax(axis=1)
        f1_tmp = f1_score(y_true, preds_tmp, average="macro")
        if f1_tmp > best_f1:
            best_f1         = f1_tmp
            best_thresholds = np.array([0.5, 0.5, 0.5, t3])

    print(f"  Threshold search best macro-F1 : {best_f1:.4f}  (CRITICAL t={best_thresholds[3]:.2f})")
    return best_thresholds

best_thresh = search_thresholds(y_prob, y_val)

# Final adjusted predictions
adjusted_prob = y_prob.copy()
adjusted_prob[:, 3] *= (0.5 / (best_thresh[3] + EPS))
y_pred_adj = adjusted_prob.argmax(axis=1)



── Training final model ──
Training until validation scores don't improve for 100 rounds
[50]	valid_0's multi_logloss: 0.268837
[100]	valid_0's multi_logloss: 0.249619
[150]	valid_0's multi_logloss: 0.248736
[200]	valid_0's multi_logloss: 0.249139
[250]	valid_0's multi_logloss: 0.250002
Early stopping, best iteration is:
[158]	valid_0's multi_logloss: 0.248562
  Threshold search best macro-F1 : 0.9089  (CRITICAL t=0.45)


In [10]:
# ============================================================================
# 9. EVALUATION
# ============================================================================
print("\n" + "=" * 60)
print("EVALUATION — FINAL MODEL")
print("=" * 60)

# ── 9a. Per-class classification report ─────────────────────────────────────
print("\nClassification report (default threshold):")
print(classification_report(
    y_val, y_pred,
    target_names=[LABEL_MAP[i] for i in range(4)]
))

print("Classification report (adjusted threshold):")
print(classification_report(
    y_val, y_pred_adj,
    target_names=[LABEL_MAP[i] for i in range(4)]
))

# ── 9b. OVR AUC-ROC ─────────────────────────────────────────────────────────
auc_ovr = roc_auc_score(y_val, y_prob, multi_class="ovr", average="macro")
auc_ovo = roc_auc_score(y_val, y_prob, multi_class="ovo", average="macro")
print(f"Macro AUC-ROC  OVR : {auc_ovr:.4f}   OVO : {auc_ovo:.4f}")

# ── 9c. Confusion matrix heatmap ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (preds, title) in zip(axes, [
    (y_pred,     "Default threshold"),
    (y_pred_adj, "Adjusted threshold (CRITICAL boosted)"),
]):
    cm = confusion_matrix(y_val, preds)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=[LABEL_MAP[i] for i in range(4)],
        yticklabels=[LABEL_MAP[i] for i in range(4)],
        ax=ax
    )
    ax.set_title(title)
    ax.set_ylabel("True label")
    ax.set_xlabel("Predicted label")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.close()

# ── 9d. Calibration plot (reliability diagram) ──────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for cls, ax in zip(range(4), axes.flat):
    prob_true, prob_pred = calibration_curve(
        (y_val == cls).astype(int),
        y_prob[:, cls],
        n_bins=10
    )
    ax.plot(prob_pred, prob_true, marker="o", label="Model")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect")
    ax.set_title(f"Calibration — {LABEL_MAP[cls]}")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.legend()
plt.suptitle("Reliability Diagrams (per class)", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "calibration.png", dpi=150)
plt.close()

# ── 9e. 5-fold cross-validation (final sanity check) ────────────────────────
print("\n── 5-fold CV on full training set ──")
cv_f1s, cv_aucs = [], []

for fold, (tr_idx, vl_idx) in enumerate(SKF.split(X, y)):
    Xtr, Xvl = X.iloc[tr_idx], X.iloc[vl_idx]
    ytr, yvl = y.iloc[tr_idx], y.iloc[vl_idx]

    m = lgb.LGBMClassifier(**best_params)
    m.fit(
        Xtr, ytr,
        eval_set  = [(Xvl, yvl)],
        callbacks = [lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(period=-1)],
    )
    pp = m.predict(Xvl)
    pb = m.predict_proba(Xvl)
    cv_f1s.append(f1_score(yvl, pp, average="macro"))
    cv_aucs.append(roc_auc_score(yvl, pb, multi_class="ovr", average="macro"))
    print(f"  Fold {fold+1}  macro-F1={cv_f1s[-1]:.4f}  AUC-OVR={cv_aucs[-1]:.4f}")

print(f"\n  CV macro-F1 : {np.mean(cv_f1s):.4f} ± {np.std(cv_f1s):.4f}")
print(f"  CV AUC-OVR  : {np.mean(cv_aucs):.4f} ± {np.std(cv_aucs):.4f}")


EVALUATION — FINAL MODEL

Classification report (default threshold):
              precision    recall  f1-score   support

         LOW       0.92      0.95      0.94      1660
      MEDIUM       0.89      0.86      0.88      1802
        HIGH       0.90      0.89      0.89      1645
    CRITICAL       0.93      0.93      0.93       893

    accuracy                           0.91      6000
   macro avg       0.91      0.91      0.91      6000
weighted avg       0.91      0.91      0.91      6000

Classification report (adjusted threshold):
              precision    recall  f1-score   support

         LOW       0.92      0.95      0.94      1660
      MEDIUM       0.89      0.86      0.88      1802
        HIGH       0.90      0.89      0.89      1645
    CRITICAL       0.93      0.94      0.93       893

    accuracy                           0.91      6000
   macro avg       0.91      0.91      0.91      6000
weighted avg       0.91      0.91      0.91      6000

Macro AUC-ROC  O

In [13]:
# ============================================================================
# 10. FEATURE IMPORTANCE
# ============================================================================
print("\n── Feature importance ──")

# ── 10a. LightGBM built-in (gain) ───────────────────────────────────────────
gain_imp = pd.Series(
    final_model.booster_.feature_importance(importance_type="gain"),
    index=X.columns
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, max(6, len(gain_imp) * 0.3)))
gain_imp.head(40).plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("LightGBM Feature Importance (Gain) — Top 40")
ax.set_xlabel("Total information gain")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_importance_gain.png", dpi=150)
plt.close()
print("  Saved: feature_importance_gain.png")

# ── 10b. SHAP summary plot ───────────────────────────────────────────────────
if SHAP_AVAILABLE:
    print("  Computing SHAP values (TreeExplainer) …")
    explainer   = shap.TreeExplainer(final_model)
    # Use a subsample for speed
    X_shap      = X_val.sample(min(3000, len(X_val)), random_state=SEED)
    shap_values = explainer.shap_values(X_shap)

    # Handle multi-class SHAP output formats:
    # - old versions: list of 4 arrays (n, p)
    # - newer versions: ndarray (n, p, 4)
    if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        shap_values = [shap_values[:, :, cls] for cls in range(shap_values.shape[2])]

    for cls in range(len(shap_values)):
        fig_shap, ax_shap = plt.subplots(figsize=(10, 8))
        shap.summary_plot(
            shap_values[cls], X_shap,
            show=False,
            plot_type="bar",
            max_display=20,
        )
        plt.title(f"SHAP Bar Plot — class {LABEL_MAP[cls]}")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"shap_bar_cls{cls}.png", dpi=150)
        plt.close()
    print("  Saved SHAP plots for all 4 classes.")
else:
    print("  Skipping SHAP (not installed).")

# ── 10c. Permutation importance (model-agnostic) ─────────────────────────────
print("  Computing permutation importance …")
perm_result = permutation_importance(
    final_model, X_val, y_val,
    n_repeats    = 10,
    random_state = SEED,
    scoring      = "f1_macro",
    n_jobs       = -1,
)
perm_imp = pd.Series(
    perm_result.importances_mean, index=X.columns
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, max(6, len(perm_imp) * 0.3)))
perm_imp.head(40).plot(kind="barh", ax=ax, color="darkorange")
ax.set_title("Permutation Importance (macro-F1 drop) — Top 40")
ax.set_xlabel("Mean drop in macro-F1")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_importance_permutation.png", dpi=150)
plt.close()
print("  Saved: feature_importance_permutation.png")

# ── Print top 20 by both methods ─────────────────────────────────────────────
summary_df = pd.DataFrame({
    "Gain_rank"       : gain_imp.rank(ascending=False).astype(int),
    "Perm_rank"       : perm_imp.rank(ascending=False).astype(int),
    "Gain_importance" : gain_imp,
    "Perm_importance" : perm_imp,
}).sort_values("Gain_rank")

print("\nTop 20 features by Gain importance:")
print(summary_df.head(20).to_string())



── Feature importance ──
  Saved: feature_importance_gain.png
  Computing SHAP values (TreeExplainer) …
  Saved SHAP plots for all 4 classes.
  Computing permutation importance …
  Saved: feature_importance_permutation.png

Top 20 features by Gain importance:
                             Gain_rank  Perm_rank  Gain_importance  Perm_importance
events_near_border                   1          3     85869.692020         0.032729
border_signal_interaction            2          5     59387.759607         0.010812
log_events_near_border               3          6     49297.791838         0.010494
border_vs_region_mean                4          4     48882.523916         0.011224
log_total_events                     5         12     43810.996039         0.001745
n_active_scenarios                   6          1     33858.044497         0.117428
log_weighted_signal                  7         13     32189.024530         0.000988
border_scenario_interaction          8          9     29315.265903 

In [14]:
# ============================================================================
# 11. SAVE ARTEFACTS
# ============================================================================
print("\n── Saving artefacts ──")

# ── 11a. Model ───────────────────────────────────────────────────────────────
model_path = OUTPUT_DIR / "model5_lgbm.txt"
final_model.booster_.save_model(str(model_path))
print(f"  Model saved  : {model_path}")

# ── 11b. Feature list (column order matters at inference) ────────────────────
feat_path = OUTPUT_DIR / "feature_columns.txt"
feat_path.write_text("\n".join(X.columns.tolist()))
print(f"  Feature list : {feat_path}")

# ── 11c. Feature importance CSVs ─────────────────────────────────────────────
summary_df.to_csv(OUTPUT_DIR / "feature_importance_summary.csv")

# ── 11d. Best hyper-parameters ───────────────────────────────────────────────
import json
hp_path = OUTPUT_DIR / "best_hyperparams.json"
hp_path.write_text(json.dumps(study.best_params, indent=2))
print(f"  Hyperparams  : {hp_path}")

# ── 11e. Summary metrics ─────────────────────────────────────────────────────
metrics = {
    "baseline_macro_f1"  : round(base_f1, 4),
    "final_macro_f1"     : round(f1_score(y_val, y_pred, average="macro"), 4),
    "final_adj_macro_f1" : round(f1_score(y_val, y_pred_adj, average="macro"), 4),
    "auc_ovr"            : round(auc_ovr, 4),
    "auc_ovo"            : round(auc_ovo, 4),
    "cv_macro_f1_mean"   : round(float(np.mean(cv_f1s)), 4),
    "cv_macro_f1_std"    : round(float(np.std(cv_f1s)),  4),
    "cv_auc_mean"        : round(float(np.mean(cv_aucs)), 4),
    "best_critical_threshold" : round(float(best_thresh[3]), 2),
    "n_features"         : int(X.shape[1]),
    "n_train"            : int(X_train.shape[0]),
    "n_val"              : int(X_val.shape[0]),
    "timestamp"          : datetime.now().isoformat(),
}
metrics_path = OUTPUT_DIR / "metrics_summary.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"  Metrics      : {metrics_path}")

print("\n" + "=" * 60)
print("ALL DONE")
print("=" * 60)
print(f"\nFinal model macro-F1 (val)          : {metrics['final_macro_f1']}")
print(f"Adj. macro-F1 (CRITICAL threshold)  : {metrics['final_adj_macro_f1']}")
print(f"AUC-ROC OVR                         : {metrics['auc_ovr']}")
print(f"5-fold CV macro-F1                  : {metrics['cv_macro_f1_mean']} ± {metrics['cv_macro_f1_std']}")
print(f"\nOutputs in : {OUTPUT_DIR.resolve()}")


── Saving artefacts ──
  Model saved  : model5_outputs/model5_lgbm.txt
  Feature list : model5_outputs/feature_columns.txt
  Hyperparams  : model5_outputs/best_hyperparams.json
  Metrics      : model5_outputs/metrics_summary.json

ALL DONE

Final model macro-F1 (val)          : 0.9087
Adj. macro-F1 (CRITICAL threshold)  : 0.9089
AUC-ROC OVR                         : 0.9861
5-fold CV macro-F1                  : 0.905 ± 0.0031

Outputs in : /Users/amitsingh/ML_Projects/WarfareAI/notebook/cybint/model5_outputs
